In [4]:
import pandas as pd
import numpy as np  

df = pd.read_csv('d:/PENS-EEPIS/SDT A Semester 4 2026/Project TA/model/dataset/raw/DataCoSupplyChainDataset.csv', encoding='latin-1')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [6]:
from pathlib import Path

# Feature engineering khusus demand forecasting.
# Target demand = total kuantitas item yang dipesan per periode waktu.
# Kolom post-order, revenue/profit, status pengiriman, dan identitas customer tidak dipakai
# agar tidak menyebabkan data leakage.

DATE_COL = 'order date (DateOrders)'
TARGET_COL = 'Order Item Quantity'
FREQ = 'W-MON'

GROUP_COLS = [
    'Product Card Id',
    'Product Name',
    'Product Category Id',
    'Category Id',
    'Category Name',
    'Department Id',
    'Department Name'
]

FORECASTING_SAFE_COLS = [DATE_COL, TARGET_COL] + GROUP_COLS
dropped_cols = [col for col in df.columns if col not in FORECASTING_SAFE_COLS]

print(f'Jumlah kolom awal: {df.shape[1]}')
print(f'Jumlah kolom yang dipakai untuk forecasting: {len(FORECASTING_SAFE_COLS)}')
print(f'Jumlah kolom yang di-drop karena tidak relevan / berisiko leakage: {len(dropped_cols)}')
print(dropped_cols)

Jumlah kolom awal: 53
Jumlah kolom yang dipakai untuk forecasting: 9
Jumlah kolom yang di-drop karena tidak relevan / berisiko leakage: 44
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Description', 'Product Image', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']


In [7]:
forecast_source = df[FORECASTING_SAFE_COLS].copy()
forecast_source[DATE_COL] = pd.to_datetime(forecast_source[DATE_COL], errors='coerce')
forecast_source = forecast_source.dropna(subset=[DATE_COL, TARGET_COL])
forecast_source[TARGET_COL] = forecast_source[TARGET_COL].astype(float)

# Agregasi transaksi menjadi deret waktu demand mingguan per produk.
demand_weekly = (
    forecast_source
    .groupby([pd.Grouper(key=DATE_COL, freq=FREQ), *GROUP_COLS], observed=True)[TARGET_COL]
    .sum()
    .reset_index()
    .rename(columns={DATE_COL: 'ds', TARGET_COL: 'demand'})
)

demand_weekly = demand_weekly.sort_values(GROUP_COLS + ['ds']).reset_index(drop=True)

# Lengkapi periode yang hilang per produk dengan demand = 0.
# Ini aman untuk forecasting karena hanya membentuk kalender historis, bukan memakai data masa depan sebagai fitur.
date_range = pd.date_range(demand_weekly['ds'].min(), demand_weekly['ds'].max(), freq=FREQ)
product_index = demand_weekly[GROUP_COLS].drop_duplicates().assign(_key=1)
calendar_index = pd.DataFrame({'ds': date_range, '_key': 1})

full_index = product_index.merge(calendar_index, on='_key').drop(columns='_key')

forecast_df = (
    full_index
    .merge(demand_weekly, on=GROUP_COLS + ['ds'], how='left')
    .sort_values(GROUP_COLS + ['ds'])
    .reset_index(drop=True)
)
forecast_df['demand'] = forecast_df['demand'].fillna(0)

In [8]:
# Fitur kalender aman karena tersedia sebelum periode prediksi.
forecast_df['year'] = forecast_df['ds'].dt.year
forecast_df['month'] = forecast_df['ds'].dt.month
forecast_df['quarter'] = forecast_df['ds'].dt.quarter
forecast_df['weekofyear'] = forecast_df['ds'].dt.isocalendar().week.astype(int)
forecast_df['is_month_start'] = forecast_df['ds'].dt.is_month_start.astype(int)
forecast_df['is_month_end'] = forecast_df['ds'].dt.is_month_end.astype(int)

forecast_df['month_sin'] = np.sin(2 * np.pi * forecast_df['month'] / 12)
forecast_df['month_cos'] = np.cos(2 * np.pi * forecast_df['month'] / 12)
forecast_df['week_sin'] = np.sin(2 * np.pi * forecast_df['weekofyear'] / 52)
forecast_df['week_cos'] = np.cos(2 * np.pi * forecast_df['weekofyear'] / 52)

# Semua fitur histori memakai shift(1), sehingga nilai demand periode yang sedang diprediksi
# tidak ikut masuk ke fitur model.
series_group = forecast_df.groupby(GROUP_COLS, observed=True)['demand']

for lag in [1, 2, 3, 4, 8, 12]:
    forecast_df[f'demand_lag_{lag}'] = series_group.shift(lag)

for window in [4, 8, 12]:
    forecast_df[f'demand_rolling_mean_{window}'] = series_group.transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
    )
    forecast_df[f'demand_rolling_std_{window}'] = series_group.transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=2).std()
    )
    forecast_df[f'demand_rolling_min_{window}'] = series_group.transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=1).min()
    )
    forecast_df[f'demand_rolling_max_{window}'] = series_group.transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=1).max()
    )



In [9]:
forecast_df['demand_expanding_mean'] = series_group.transform(
    lambda s: s.shift(1).expanding(min_periods=1).mean()
)

# Missing value dari awal histori diisi 0 karena belum ada riwayat sebelum periode tersebut.
history_feature_cols = [col for col in forecast_df.columns if col.startswith('demand_lag_') or col.startswith('demand_rolling_')]
history_feature_cols.append('demand_expanding_mean')
forecast_df[history_feature_cols] = forecast_df[history_feature_cols].fillna(0)

# Urutan kolom dibuat jelas: identitas seri, waktu, target, lalu fitur model.
ordered_cols = GROUP_COLS + ['ds', 'demand'] + [
    col for col in forecast_df.columns if col not in GROUP_COLS + ['ds', 'demand']
]
forecast_df = forecast_df[ordered_cols]

output_path = Path('D:/PENS-EEPIS/SDT A Semester 4 2026/Project TA/model/dataset/engineered/demand_forecasting_features.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
forecast_df.to_csv(output_path, index=False)

print(f'Ukuran dataset forecasting: {forecast_df.shape}')
print(f'File tersimpan di: {output_path}')

forecast_df.head()

Ukuran dataset forecasting: (19116, 38)
File tersimpan di: D:\PENS-EEPIS\SDT A Semester 4 2026\Project TA\model\dataset\engineered\demand_forecasting_features.csv


,Product Card Id,Product Name,Product Category Id,Category Id,Category Name,Department Id,Department Name,ds,demand,year,...,demand_rolling_max_4,demand_rolling_mean_8,demand_rolling_std_8,demand_rolling_min_8,demand_rolling_max_8,demand_rolling_mean_12,demand_rolling_std_12,demand_rolling_min_12,demand_rolling_max_12,demand_expanding_mean
0,19,Nike Men's Fingertrap Max Training Shoe,2,2,Soccer,2,Fitness,2015-01-05,0.0,2015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,19,Nike Men's Fingertrap Max Training Shoe,2,2,Soccer,2,Fitness,2015-01-12,0.0,2015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,19,Nike Men's Fingertrap Max Training Shoe,2,2,Soccer,2,Fitness,2015-01-19,0.0,2015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,19,Nike Men's Fingertrap Max Training Shoe,2,2,Soccer,2,Fitness,2015-01-26,0.0,2015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,19,Nike Men's Fingertrap Max Training Shoe,2,2,Soccer,2,Fitness,2015-02-02,0.0,2015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
forecast_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19116 entries, 0 to 19115
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Product Card Id         19116 non-null  int64         
 1   Product Name            19116 non-null  object        
 2   Product Category Id     19116 non-null  int64         
 3   Category Id             19116 non-null  int64         
 4   Category Name           19116 non-null  object        
 5   Department Id           19116 non-null  int64         
 6   Department Name         19116 non-null  object        
 7   ds                      19116 non-null  datetime64[ns]
 8   demand                  19116 non-null  float64       
 9   year                    19116 non-null  int32         
 10  month                   19116 non-null  int32         
 11  quarter                 19116 non-null  int32         
 12  weekofyear              19116 non-null  int32 